# Phase 4: QC Warm-Up Experiment

**Version: 1.2** | 2026-03-09

**Question:** Does conditioning the LSTM hidden state with QC injection sequences improve next-m/z-bin prediction on analytical samples?

**Mechanism:** Feed QC elution sequences through the LSTM using teacher forcing (ground truth tokens) to build up hidden state, then evaluate on analytical samples starting from this warmed-up state instead of zeros.

**Design:** Option B (pragmatic) — use existing model, exclude QC samples that were in training from the conditioning pool.

**Experiment:**
- Dose-response: N = 0, 2, 4, 6, 8, 10 QC injections
- Two evaluation modes:
  - **Prime-only:** Warm up hidden state with QCs, use it for the first prediction only, then revert to standard sliding window
  - **Carry-hidden:** Warm up with QCs, then propagate hidden state through the entire analytical sample
- Control 1: Analytical sample warm-up (non-QC conditioning)
- Control 2: Cross-cohort QC warm-up (QCs from different study)
- Control 3: Shuffled QC warm-up (destroy temporal structure)

**Prerequisites:** Run `01_train_models.ipynb` first — this notebook loads saved LSTM checkpoint.

**Monitoring:** Progress is logged to `outputs/qc_warmup_log.csv` on Google Drive after each condition. Monitor remotely via Google Drive Desktop.

**Changelog:**
- v1.2: Add incremental CSV logging to Drive after each condition (crash-resilient)
- v1.1: Add carry-hidden evaluation mode (propagate hidden state through entire analytical sample)
- v1.0: Split from `02_post_training_experiments.ipynb` into standalone notebook

## 1. Setup

In [ ]:
import subprocess, sys, os, copy

import torch
import torch.nn as nn
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    import pyarrow
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyarrow"])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# === Paths ===
BASE_DIR = "/content/drive/MyDrive/0000 Fun with coding/088 Lights-Out R01 Grant/Specific Aim 1/poc3_elution_sequence"
EXPERIMENT_DIR = f"{BASE_DIR}/03_qc_warmup"
OUTPUT_DIR = f"{EXPERIMENT_DIR}/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Model checkpoint (from training notebook — in 01_train_models subfolder) — LSTM only (warm-up uses hidden state)
TRAIN_DIR = f"{BASE_DIR}/01_train_models"
LSTM_CHECKPOINT = f"{TRAIN_DIR}/outputs/lstm_best.pt"

# Data
TOKENIZED_PATH = f"{TRAIN_DIR}/tokenized_features.parquet"

print(f"Experiment dir: {EXPERIMENT_DIR}")
print(f"LSTM checkpoint: {LSTM_CHECKPOINT}")
print(f"Training data: {TOKENIZED_PATH}")

# === Incremental logging (crash-resilient) ===
import csv, time
from datetime import datetime

LOG_PATH = f"{OUTPUT_DIR}/qc_warmup_log.csv"
LOG_FIELDS = ["timestamp", "phase", "condition", "n_qc", "n_predictions",
              "top1", "top3", "top5", "top10", "mae_da", "elapsed_s", "status"]

def log_condition(phase, condition, n_qc, results_df, elapsed, status="DONE"):
    """Append one row to the progress log on Drive."""
    file_exists = os.path.exists(LOG_PATH)
    row = {
        "timestamp": datetime.now().isoformat(),
        "phase": phase,
        "condition": condition,
        "n_qc": n_qc,
        "n_predictions": len(results_df) if len(results_df) > 0 else 0,
        "top1": f"{results_df['top1'].mean():.6f}" if len(results_df) > 0 else "",
        "top3": f"{results_df['top3'].mean():.6f}" if len(results_df) > 0 and 'top3' in results_df else "",
        "top5": f"{results_df['top5'].mean():.6f}" if len(results_df) > 0 else "",
        "top10": f"{results_df['top10'].mean():.6f}" if len(results_df) > 0 and 'top10' in results_df else "",
        "mae_da": f"{results_df['mae_da'].mean():.1f}" if len(results_df) > 0 else "",
        "elapsed_s": f"{elapsed:.1f}",
        "status": status,
    }
    with open(LOG_PATH, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=LOG_FIELDS)
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)

# Log experiment start
log_condition("setup", "start", 0, pd.DataFrame(), 0, "STARTED")
print(f"Progress log: {LOG_PATH}")
experiment_start = time.time()

In [ ]:
# === Configuration (must match training notebook) ===
RANDOM_SEED = 42
CONTEXT_LENGTH = 64
EMBEDDING_DIM = 64
HIDDEN_DIM = 128
NUM_LAYERS = 2
DROPOUT = 0.1
MZ_BIN_WIDTH = 10  # Da

RT_GAP_LABELS = ["co-elute", "0.1-0.5s", "0.5-1s", "1-2s", "2-5s", "5-15s", ">15s"]
INTENSITY_RANK_LABELS = ["top1%", "top5%", "top20%", "top50%", "low"]
POLARITY_MAP = {"pos": 0, "neg": 1, "unk": 2}
RT_GAP_MAP = {label: i for i, label in enumerate(RT_GAP_LABELS)}
INTENSITY_MAP = {label: i for i, label in enumerate(INTENSITY_RANK_LABELS)}
TEST_FRACTION = 0.15
VAL_FRACTION = 0.15

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

## 2. Model Definition & Checkpoint Loading

In [ ]:
class MultiFieldEmbedding(nn.Module):
    def __init__(self, embedding_dim, max_mz_bin=120, max_md_bin=20,
                 max_rt_gap=7, max_polarity=3, max_intensity=5):
        super().__init__()
        self.mz_embed = nn.Embedding(max_mz_bin, embedding_dim)
        self.md_embed = nn.Embedding(max_md_bin, embedding_dim)
        self.gap_embed = nn.Embedding(max_rt_gap, embedding_dim)
        self.pol_embed = nn.Embedding(max_polarity, embedding_dim)
        self.int_embed = nn.Embedding(max_intensity, embedding_dim)

    def forward(self, mz, md, gap, pol, intensity):
        return (self.mz_embed(mz) + self.md_embed(md) + self.gap_embed(gap)
                + self.pol_embed(pol) + self.int_embed(intensity))


class LSTMModel(nn.Module):
    """LSTM with optional hidden state input for warm-up experiments."""
    def __init__(self, num_mz_classes, embedding_dim=64, hidden_dim=128,
                 num_layers=2, dropout=0.1, **embed_kw):
        super().__init__()
        self.embedding = MultiFieldEmbedding(embedding_dim, **embed_kw)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers,
                           dropout=dropout if num_layers > 1 else 0, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_dim, num_mz_classes)

    def forward(self, mz, md, gap, pol, intensity, hidden=None):
        x = self.embedding(mz, md, gap, pol, intensity)
        out, hidden = self.lstm(x, hidden)
        return self.head(self.dropout(out[:, -1, :])), hidden

In [ ]:
# === Load trained LSTM checkpoint ===
lstm_ckpt = torch.load(LSTM_CHECKPOINT, map_location=device, weights_only=False)
MAX_MZ_BIN = lstm_ckpt["config"]["num_classes"]
print(f"Max m/z bin: {MAX_MZ_BIN}")

embed_kw = dict(max_mz_bin=MAX_MZ_BIN, max_md_bin=20, max_rt_gap=7, max_polarity=3, max_intensity=5)

lstm_model = LSTMModel(MAX_MZ_BIN, EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT, **embed_kw)
lstm_model.load_state_dict(lstm_ckpt["model_state_dict"])
lstm_model = lstm_model.to(device)
lstm_model.eval()
print(f"LSTM loaded \u2014 test metrics: {lstm_ckpt['test_metrics']}")

## 3. Prepare QC and Analytical Sample Pools

In [ ]:
# === Load training data ===
tok = pd.read_parquet(TOKENIZED_PATH)
print(f"Training data: {len(tok):,} rows, {tok[['study','sample_id']].drop_duplicates().shape[0]} samples")

# Encode categorical fields
tok["polarity_idx"] = tok["polarity_tok"].map(POLARITY_MAP).fillna(2).astype(int)
tok["rt_gap_idx"] = tok["rt_gap_bin"].astype(str).map(RT_GAP_MAP).fillna(0).astype(int)
tok["intensity_idx"] = tok["intensity_rank"].astype(str).map(INTENSITY_MAP).fillna(4).astype(int)
tok = tok.sort_values(["study", "sample_id", "seq_pos"])

In [ ]:
def build_sample_arrays(df):
    """Convert DataFrame to list of per-sample numpy arrays."""
    samples = []
    for (study, sid), group in df.groupby(["study", "sample_id"]):
        g = group.sort_values("seq_pos")
        samples.append({
            "study": study, "sample_id": sid,
            "sample_type": g["sample_type"].iloc[0] if "sample_type" in g.columns else "analytical",
            "mz_bin": g["mz_bin"].values.astype(np.int64),
            "md_bin": g["md_bin"].values.astype(np.int64),
            "rt_gap_idx": g["rt_gap_idx"].values.astype(np.int64),
            "polarity_idx": g["polarity_idx"].values.astype(np.int64),
            "intensity_idx": g["intensity_idx"].values.astype(np.int64),
        })
    return samples

In [ ]:
# === Identify QC and analytical samples, rebuild train/val/test split ===
sample_meta = tok[["study", "sample_id", "sample_type"]].drop_duplicates()
print("Sample counts by study and type:")
print(sample_meta.groupby(["study", "sample_type"]).size().unstack(fill_value=0))

rng = np.random.RandomState(RANDOM_SEED)
samples = tok[["study", "sample_id"]].drop_duplicates().reset_index(drop=True)

train_keys, val_keys, test_keys = set(), set(), set()
for study, group in samples.groupby("study"):
    indices = group.index.values.copy()
    rng.shuffle(indices)
    n = len(indices)
    n_test = max(1, int(n * TEST_FRACTION))
    n_val = max(1, int(n * VAL_FRACTION))
    for idx in indices[:n_test]:
        test_keys.add((group.loc[idx, "study"], group.loc[idx, "sample_id"]))
    for idx in indices[n_test:n_test + n_val]:
        val_keys.add((group.loc[idx, "study"], group.loc[idx, "sample_id"]))
    for idx in indices[n_test + n_val:]:
        train_keys.add((group.loc[idx, "study"], group.loc[idx, "sample_id"]))

# QC samples NOT in training (available for conditioning)
qc_meta = sample_meta[sample_meta.sample_type == "QC"]
qc_in_train = qc_meta.apply(lambda r: (r["study"], r["sample_id"]) in train_keys, axis=1)
qc_available = qc_meta[~qc_in_train]
print(f"\nQC samples in training: {qc_in_train.sum()}")
print(f"QC samples available for conditioning: {len(qc_available)}")
print(qc_available.groupby("study").size())

# Test analytical samples
test_analytical = sample_meta[
    (sample_meta.sample_type == "analytical") &
    sample_meta.apply(lambda r: (r["study"], r["sample_id"]) in test_keys, axis=1)
]
print(f"\nTest analytical samples: {len(test_analytical)}")
print(test_analytical.groupby("study").size())

In [ ]:
# === Build sample arrays per cohort ===
# QC arrays (for conditioning)
qc_arrays = {}
for study in tok["study"].unique():
    qc_ids = qc_available[qc_available.study == study]["sample_id"].tolist()
    if not qc_ids:
        continue
    qc_df = tok[(tok.study == study) & (tok.sample_id.isin(qc_ids))]
    qc_arrays[study] = build_sample_arrays(qc_df)
    print(f"  {study}: {len(qc_arrays[study])} QC samples for conditioning")

# Test analytical arrays
test_arrays = {}
for study in tok["study"].unique():
    test_ids = test_analytical[test_analytical.study == study]["sample_id"].tolist()
    if not test_ids:
        continue
    test_df = tok[(tok.study == study) & (tok.sample_id.isin(test_ids))]
    test_arrays[study] = build_sample_arrays(test_df)
    print(f"  {study}: {len(test_arrays[study])} test analytical samples")

# Analytical conditioning pool (train analytical, not QC — for Control 1)
analytical_arrays = {}
for study in tok["study"].unique():
    train_analytical_ids = sample_meta[
        (sample_meta.study == study) & 
        (sample_meta.sample_type == "analytical") &
        sample_meta.apply(lambda r: (r["study"], r["sample_id"]) in train_keys, axis=1)
    ]["sample_id"].tolist()
    if train_analytical_ids:
        ana_df = tok[(tok.study == study) & (tok.sample_id.isin(train_analytical_ids))]
        analytical_arrays[study] = build_sample_arrays(ana_df)
        print(f"  {study}: {len(analytical_arrays[study])} train analytical samples (for control)")

## 4. QC Warm-Up Evaluation Functions

In [ ]:
def lstm_warmup_evaluate(model, qc_samples, test_sample, n_qc, context_length=CONTEXT_LENGTH):
    """
    Evaluate LSTM on a test sample after warming up hidden state with N QC samples.
    Prime-only mode: warmed-up hidden used at first position, then standard sliding window.
    
    Args:
        model: Trained LSTM model
        qc_samples: List of QC sample arrays (from same cohort)
        test_sample: Single test analytical sample array
        n_qc: Number of QC samples to use for warm-up (0 = cold start)
    
    Returns:
        List of per-position result dicts
    """
    model.eval()
    results = []
    
    with torch.no_grad():
        # === Phase 1: Warm up hidden state with QC sequences ===
        hidden = None  # Start with zeros
        
        if n_qc > 0 and qc_samples:
            qc_to_use = qc_samples[:n_qc]
            for qc in qc_to_use:
                n_qc_features = len(qc["mz_bin"])
                for pos in range(0, n_qc_features, context_length):
                    end = min(pos + context_length, n_qc_features)
                    
                    mz = torch.tensor(qc["mz_bin"][pos:end], dtype=torch.long).unsqueeze(0).to(device)
                    md = torch.tensor(qc["md_bin"][pos:end], dtype=torch.long).unsqueeze(0).to(device)
                    gap = torch.tensor(qc["rt_gap_idx"][pos:end], dtype=torch.long).unsqueeze(0).to(device)
                    pol = torch.tensor(qc["polarity_idx"][pos:end], dtype=torch.long).unsqueeze(0).to(device)
                    inten = torch.tensor(qc["intensity_idx"][pos:end], dtype=torch.long).unsqueeze(0).to(device)
                    
                    _, hidden = model(mz, md, gap, pol, inten, hidden)
        
        # === Phase 2: Evaluate on analytical sample ===
        n_features = len(test_sample["mz_bin"])
        if n_features <= context_length:
            return results
        
        for pos in range(context_length, n_features):
            start = pos - context_length
            mz = torch.tensor(test_sample["mz_bin"][start:pos], dtype=torch.long).unsqueeze(0).to(device)
            md = torch.tensor(test_sample["md_bin"][start:pos], dtype=torch.long).unsqueeze(0).to(device)
            gap = torch.tensor(test_sample["rt_gap_idx"][start:pos], dtype=torch.long).unsqueeze(0).to(device)
            pol = torch.tensor(test_sample["polarity_idx"][start:pos], dtype=torch.long).unsqueeze(0).to(device)
            inten = torch.tensor(test_sample["intensity_idx"][start:pos], dtype=torch.long).unsqueeze(0).to(device)
            target = int(test_sample["mz_bin"][pos])
            
            # First prediction uses warmed-up hidden; subsequent use sliding window (no hidden carry)
            if pos == context_length:
                logits, _ = model(mz, md, gap, pol, inten, hidden)
            else:
                logits, _ = model(mz, md, gap, pol, inten)
            
            pred = logits.argmax(dim=1).item()
            topk_preds = {}
            for k in (1, 3, 5, 10):
                _, tk = logits.topk(k, dim=1)
                topk_preds[f"top{k}"] = target in tk[0].cpu().numpy()
            
            results.append({
                "study": test_sample["study"], "sample_id": test_sample["sample_id"],
                "position": pos, "target": target, "pred": pred,
                "mae_da": abs(pred - target) * MZ_BIN_WIDTH,
                **topk_preds,
            })
    
    return results


def lstm_warmup_evaluate_carry_hidden(model, qc_samples, test_sample, n_qc, context_length=CONTEXT_LENGTH):
    """
    Same as above but carries hidden state through the entire analytical sample
    (not just the first position). Tests continuous hidden state propagation.
    
    This answers: does the LSTM benefit from maintaining state across the whole
    analytical run, or is the initial prime sufficient?
    """
    model.eval()
    results = []
    
    with torch.no_grad():
        hidden = None
        
        # Warm up with QC sequences
        if n_qc > 0 and qc_samples:
            for qc in qc_samples[:n_qc]:
                n_qc_features = len(qc["mz_bin"])
                for pos in range(0, n_qc_features, context_length):
                    end = min(pos + context_length, n_qc_features)
                    mz = torch.tensor(qc["mz_bin"][pos:end], dtype=torch.long).unsqueeze(0).to(device)
                    md = torch.tensor(qc["md_bin"][pos:end], dtype=torch.long).unsqueeze(0).to(device)
                    gap = torch.tensor(qc["rt_gap_idx"][pos:end], dtype=torch.long).unsqueeze(0).to(device)
                    pol = torch.tensor(qc["polarity_idx"][pos:end], dtype=torch.long).unsqueeze(0).to(device)
                    inten = torch.tensor(qc["intensity_idx"][pos:end], dtype=torch.long).unsqueeze(0).to(device)
                    _, hidden = model(mz, md, gap, pol, inten, hidden)
        
        # Evaluate on analytical sample, carrying hidden state throughout
        n_features = len(test_sample["mz_bin"])
        if n_features <= context_length:
            return results
        
        for pos in range(context_length, n_features):
            start = pos - context_length
            mz = torch.tensor(test_sample["mz_bin"][start:pos], dtype=torch.long).unsqueeze(0).to(device)
            md = torch.tensor(test_sample["md_bin"][start:pos], dtype=torch.long).unsqueeze(0).to(device)
            gap = torch.tensor(test_sample["rt_gap_idx"][start:pos], dtype=torch.long).unsqueeze(0).to(device)
            pol = torch.tensor(test_sample["polarity_idx"][start:pos], dtype=torch.long).unsqueeze(0).to(device)
            inten = torch.tensor(test_sample["intensity_idx"][start:pos], dtype=torch.long).unsqueeze(0).to(device)
            target = int(test_sample["mz_bin"][pos])
            
            logits, hidden = model(mz, md, gap, pol, inten, hidden)
            pred = logits.argmax(dim=1).item()
            topk_preds = {}
            for k in (1, 3, 5, 10):
                _, tk = logits.topk(k, dim=1)
                topk_preds[f"top{k}"] = target in tk[0].cpu().numpy()
            
            results.append({
                "study": test_sample["study"], "sample_id": test_sample["sample_id"],
                "position": pos, "target": target, "pred": pred,
                "mae_da": abs(pred - target) * MZ_BIN_WIDTH,
                **topk_preds,
            })
    
    return results

print("QC warm-up evaluation functions defined (prime-only + carry-hidden).")

## 5. Dose-Response Experiment

In [ ]:
N_QC_VALUES = [0, 2, 4, 6, 8, 10]
all_warmup_results = []

for n_qc in N_QC_VALUES:
    t0 = time.time()
    print(f"\n--- N_QC = {n_qc} ---")
    condition_results = []
    
    for study in sorted(test_arrays.keys()):
        qc_pool = qc_arrays.get(study, [])
        n_available = len(qc_pool)
        n_use = min(n_qc, n_available)
        
        for test_sample in test_arrays[study]:
            results = lstm_warmup_evaluate(lstm_model, qc_pool, test_sample, n_use)
            for r in results:
                r["n_qc"] = n_qc
                r["n_qc_actual"] = n_use
                r["condition"] = "qc_warmup"
            condition_results.extend(results)
    
    df = pd.DataFrame(condition_results)
    elapsed = time.time() - t0
    if len(df) > 0:
        print(f"  n={len(df):,}  top1={df['top1'].mean():.4f}  top5={df['top5'].mean():.4f}  MAE={df['mae_da'].mean():.0f}Da  ({elapsed:.0f}s)")
    log_condition("dose_response", "prime_only", n_qc, df, elapsed)
    all_warmup_results.extend(condition_results)

warmup_df = pd.DataFrame(all_warmup_results)
print(f"\nTotal warm-up results: {len(warmup_df):,}")

### 5b. Carry-Hidden Dose-Response

Same experiment but propagating hidden state through the entire analytical sample, not just the first position.

## 6. Control Experiments

In [ ]:
# === Carry-hidden dose-response ===
all_carry_results = []

for n_qc in N_QC_VALUES:
    t0 = time.time()
    print(f"\n--- Carry-hidden N_QC = {n_qc} ---")
    condition_results = []
    
    for study in sorted(test_arrays.keys()):
        qc_pool = qc_arrays.get(study, [])
        n_available = len(qc_pool)
        n_use = min(n_qc, n_available)
        
        for test_sample in test_arrays[study]:
            results = lstm_warmup_evaluate_carry_hidden(lstm_model, qc_pool, test_sample, n_use)
            for r in results:
                r["n_qc"] = n_qc
                r["n_qc_actual"] = n_use
                r["condition"] = "carry_hidden"
            condition_results.extend(results)
    
    df = pd.DataFrame(condition_results)
    elapsed = time.time() - t0
    if len(df) > 0:
        print(f"  n={len(df):,}  top1={df['top1'].mean():.4f}  top5={df['top5'].mean():.4f}  MAE={df['mae_da'].mean():.0f}Da  ({elapsed:.0f}s)")
    log_condition("dose_response", "carry_hidden", n_qc, df, elapsed)
    all_carry_results.extend(condition_results)

carry_df = pd.DataFrame(all_carry_results)
print(f"\nTotal carry-hidden results: {len(carry_df):,}")

# === Compare prime-only vs carry-hidden ===
print("\n" + "=" * 70)
print(f"{'N_QC':<6} {'Prime-only top1':<18} {'Carry-hidden top1':<18} {'Delta':<10}")
print("-" * 70)
for n_qc in N_QC_VALUES:
    prime = warmup_df[warmup_df.n_qc == n_qc]["top1"].mean()
    carry = carry_df[carry_df.n_qc == n_qc]["top1"].mean()
    delta = carry - prime
    print(f"{n_qc:<6} {prime:<18.4f} {carry:<18.4f} {delta:+.4f}")
print("=" * 70)

In [ ]:
# === Control 1: Analytical sample warm-up (instead of QC) ===
print("Running Control 1: Analytical sample warm-up...")
analytical_control_results = []

for n_qc in [0, 4, 8]:
    t0 = time.time()
    for study in sorted(test_arrays.keys()):
        ana_pool = analytical_arrays.get(study, [])
        n_use = min(n_qc, len(ana_pool))
        for test_sample in test_arrays[study]:
            results = lstm_warmup_evaluate(lstm_model, ana_pool, test_sample, n_use)
            for r in results:
                r["n_qc"] = n_qc
                r["condition"] = "analytical_warmup"
            analytical_control_results.extend(results)
    elapsed = time.time() - t0
    sub = pd.DataFrame([r for r in analytical_control_results if r["n_qc"] == n_qc])
    log_condition("control", "analytical_warmup", n_qc, sub, elapsed)

ana_ctrl_df = pd.DataFrame(analytical_control_results)
print(f"Analytical control results: {len(ana_ctrl_df):,}")
for n in [0, 4, 8]:
    sub = ana_ctrl_df[ana_ctrl_df.n_qc == n]
    if len(sub) > 0:
        print(f"  N={n}: top1={sub['top1'].mean():.4f}  top5={sub['top5'].mean():.4f}")

In [ ]:
# === Control 2: Cross-cohort QC warm-up ===
print("Running Control 2: Cross-cohort QC warm-up...")
cross_cohort_results = []

studies = sorted(test_arrays.keys())
for n_qc in [0, 4, 8]:
    t0 = time.time()
    for study in studies:
        other_studies = [s for s in studies if s != study and s in qc_arrays]
        if not other_studies:
            continue
        cross_qc = qc_arrays[other_studies[0]]
        n_use = min(n_qc, len(cross_qc))
        
        for test_sample in test_arrays[study]:
            results = lstm_warmup_evaluate(lstm_model, cross_qc, test_sample, n_use)
            for r in results:
                r["n_qc"] = n_qc
                r["condition"] = "cross_cohort_qc"
            cross_cohort_results.extend(results)
    elapsed = time.time() - t0
    sub = pd.DataFrame([r for r in cross_cohort_results if r["n_qc"] == n_qc])
    log_condition("control", "cross_cohort_qc", n_qc, sub, elapsed)

cross_df = pd.DataFrame(cross_cohort_results)
print(f"Cross-cohort results: {len(cross_df):,}")
for n in [0, 4, 8]:
    sub = cross_df[cross_df.n_qc == n]
    if len(sub) > 0:
        print(f"  N={n}: top1={sub['top1'].mean():.4f}  top5={sub['top5'].mean():.4f}")

In [ ]:
# === Control 3: Shuffled QC warm-up ===
print("Running Control 3: Shuffled QC warm-up...")
shuffled_results = []

rng_shuffle = np.random.RandomState(123)

for n_qc in [0, 4, 8]:
    t0 = time.time()
    for study in sorted(test_arrays.keys()):
        qc_pool = qc_arrays.get(study, [])
        n_use = min(n_qc, len(qc_pool))
        
        shuffled_qc = []
        for qc in qc_pool[:n_use]:
            sq = copy.deepcopy(qc)
            perm = rng_shuffle.permutation(len(sq["mz_bin"]))
            for key in ["mz_bin", "md_bin", "rt_gap_idx", "polarity_idx", "intensity_idx"]:
                sq[key] = sq[key][perm]
            shuffled_qc.append(sq)
        
        for test_sample in test_arrays[study]:
            results = lstm_warmup_evaluate(lstm_model, shuffled_qc, test_sample, n_use)
            for r in results:
                r["n_qc"] = n_qc
                r["condition"] = "shuffled_qc"
            shuffled_results.extend(results)
    elapsed = time.time() - t0
    sub = pd.DataFrame([r for r in shuffled_results if r["n_qc"] == n_qc])
    log_condition("control", "shuffled_qc", n_qc, sub, elapsed)

shuffled_df = pd.DataFrame(shuffled_results)
print(f"Shuffled QC results: {len(shuffled_df):,}")
for n in [0, 4, 8]:
    sub = shuffled_df[shuffled_df.n_qc == n]
    if len(sub) > 0:
        print(f"  N={n}: top1={sub['top1'].mean():.4f}  top5={sub['top5'].mean():.4f}")

## 7. Figure: QC Warm-Up Results

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel A: Dose-response — prime-only vs carry-hidden
ax = axes[0, 0]
for label, df, color, ls in [("Prime-only", warmup_df, "tab:blue", "-"),
                               ("Carry-hidden", carry_df, "tab:red", "--")]:
    dose = df.groupby("n_qc")["top1"].agg(["mean", "std"]).reset_index()
    ax.errorbar(dose["n_qc"], dose["mean"],
                yerr=dose["std"] / np.sqrt(dose["mean"].count()),
                marker="o", capsize=4, color=color, linewidth=2, linestyle=ls, label=label)
ax.set_xlabel("Number of QC injections")
ax.set_ylabel("Top-1 accuracy")
ax.set_title("A. Dose Response: Prime-Only vs Carry-Hidden")
ax.legend()
ax.grid(alpha=0.3)

# Panel B: Condition comparison at N=8
ax = axes[0, 1]
conditions = {
    "Cold start\n(N=0)": warmup_df[warmup_df.n_qc == 0]["top1"].mean(),
    "QC prime\n(N=8)": warmup_df[warmup_df.n_qc == 8]["top1"].mean() if 8 in warmup_df.n_qc.values else 0,
    "QC carry\n(N=8)": carry_df[carry_df.n_qc == 8]["top1"].mean() if 8 in carry_df.n_qc.values else 0,
    "Analytical\n(N=8)": ana_ctrl_df[ana_ctrl_df.n_qc == 8]["top1"].mean() if len(ana_ctrl_df) > 0 else 0,
    "Cross-cohort\n(N=8)": cross_df[cross_df.n_qc == 8]["top1"].mean() if len(cross_df) > 0 else 0,
    "Shuffled\n(N=8)": shuffled_df[shuffled_df.n_qc == 8]["top1"].mean() if len(shuffled_df) > 0 else 0,
}
colors = ["gray", "tab:blue", "tab:red", "tab:green", "tab:orange", "tab:purple"]
bars = ax.bar(range(len(conditions)), list(conditions.values()), color=colors, alpha=0.8)
ax.set_xticks(range(len(conditions)))
ax.set_xticklabels(list(conditions.keys()), fontsize=8)
ax.set_ylabel("Top-1 accuracy")
ax.set_title("B. Condition Comparison")
ax.grid(axis="y", alpha=0.3)

# Panel C: Position-dependent — prime-only
ax = axes[1, 0]
for n_qc, color, label in [(0, "gray", "Cold start"), (4, "tab:cyan", "N=4 prime"), (8, "tab:blue", "N=8 prime")]:
    sub = warmup_df[warmup_df.n_qc == n_qc].copy()
    if len(sub) == 0:
        continue
    sub["pos_bin"] = (sub["position"] // 50) * 50
    pos_acc = sub.groupby("pos_bin")["top1"].mean()
    ax.plot(pos_acc.index, pos_acc.values, label=label, color=color, alpha=0.8)
# Add carry-hidden N=8
sub = carry_df[carry_df.n_qc == 8].copy()
if len(sub) > 0:
    sub["pos_bin"] = (sub["position"] // 50) * 50
    pos_acc = sub.groupby("pos_bin")["top1"].mean()
    ax.plot(pos_acc.index, pos_acc.values, label="N=8 carry", color="tab:red", alpha=0.8, linestyle="--")
ax.set_xlabel("Position in analytical sequence")
ax.set_ylabel("Top-1 accuracy")
ax.set_title("C. Accuracy vs Position")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Panel D: Carry-hidden benefit over position (delta accuracy)
ax = axes[1, 1]
for n_qc, color in [(4, "tab:cyan"), (8, "tab:blue")]:
    prime_sub = warmup_df[warmup_df.n_qc == n_qc].copy()
    carry_sub = carry_df[carry_df.n_qc == n_qc].copy()
    if len(prime_sub) == 0 or len(carry_sub) == 0:
        continue
    prime_sub["pos_bin"] = (prime_sub["position"] // 50) * 50
    carry_sub["pos_bin"] = (carry_sub["position"] // 50) * 50
    prime_acc = prime_sub.groupby("pos_bin")["top1"].mean()
    carry_acc = carry_sub.groupby("pos_bin")["top1"].mean()
    common = prime_acc.index.intersection(carry_acc.index)
    delta = carry_acc[common] - prime_acc[common]
    ax.plot(common, delta.values, label=f"N={n_qc}", color=color, alpha=0.8)
ax.axhline(0, color="gray", linestyle=":", alpha=0.5)
ax.set_xlabel("Position in analytical sequence")
ax.set_ylabel("Carry - Prime accuracy delta")
ax.set_title("D. Carry-Hidden Benefit Over Position")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/phase4_qc_warmup.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUTPUT_DIR}/phase4_qc_warmup.png")

## 8. Per-Cohort Breakdown

In [ ]:
print("\n" + "=" * 80)
print("Per-Cohort QC Warm-Up Results")
print("=" * 80)

for study in sorted(warmup_df.study.unique()):
    print(f"\n{study}:")
    for n_qc in N_QC_VALUES:
        sub = warmup_df[(warmup_df.study == study) & (warmup_df.n_qc == n_qc)]
        if len(sub) > 0:
            print(f"  N_QC={n_qc}: n={len(sub):>5}  top1={sub['top1'].mean():.4f}  "
                  f"top5={sub['top5'].mean():.4f}  MAE={sub['mae_da'].mean():.0f}Da")

## 9. Save Results

In [ ]:
all_results = {
    "experiment": "Phase 4: QC Warm-Up",
    "prime_only_dose_response": warmup_df.groupby("n_qc").agg(
        top1=("top1", "mean"), top5=("top5", "mean"), mae_da=("mae_da", "mean"), n=("top1", "count")
    ).to_dict("index"),
    "carry_hidden_dose_response": carry_df.groupby("n_qc").agg(
        top1=("top1", "mean"), top5=("top5", "mean"), mae_da=("mae_da", "mean"), n=("top1", "count")
    ).to_dict("index"),
    "controls": {
        "analytical_warmup": ana_ctrl_df.groupby("n_qc")["top1"].mean().to_dict() if len(ana_ctrl_df) > 0 else {},
        "cross_cohort": cross_df.groupby("n_qc")["top1"].mean().to_dict() if len(cross_df) > 0 else {},
        "shuffled": shuffled_df.groupby("n_qc")["top1"].mean().to_dict() if len(shuffled_df) > 0 else {},
    },
}

with open(f"{OUTPUT_DIR}/phase4_results.json", "w") as f:
    json.dump(all_results, f, indent=2, default=str)

warmup_df.to_parquet(f"{OUTPUT_DIR}/phase4_prime_only_detailed.parquet", index=False)
carry_df.to_parquet(f"{OUTPUT_DIR}/phase4_carry_hidden_detailed.parquet", index=False)

# Log completion
total_elapsed = time.time() - experiment_start
log_condition("all", "complete", 0, pd.DataFrame(), total_elapsed, "COMPLETED")

print("All results saved.")
print(f"Total experiment time: {total_elapsed/60:.1f} minutes")
print(f"Output directory: {OUTPUT_DIR}")
for f_name in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(f"{OUTPUT_DIR}/{f_name}")
    print(f"  {f_name:<45} {size/1024:.0f} KB")